In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# BRONZE — physical_lojas
# Squad 3 — Arquitetura Medalhao
# Regra: copia fiel do dado bruto
#        sem alteracoes, sem tratamentos
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════


In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

SOURCE_FILE  = "physical_lojas.csv"
SOURCE_PATH  = f"{RAW_BATCH_PATH}{SOURCE_FILE}"
BRONZE_TABLE = "physical_lojas"
BRONZE_PATH  = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

EXPECTED_COLUMNS = [
    "id_loja",
    "nome_loja",
    "cnpj",
    "cidade_loja",
    "estado_loja",
    "peso_vendas",
]

KEY_COLUMNS = ["id_loja"]

# physical_lojas e dado estatico — sem particao por data
BRONZE_WRITE_MODE = "overwrite"

print("Constantes configuradas:")
print(f"   SOURCE_PATH  : {SOURCE_PATH}")
print(f"   BRONZE_PATH  : {BRONZE_PATH}")
print(f"   KEY_COLUMNS  : {KEY_COLUMNS}")

In [0]:
# configuracoes do ADLS  

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler dados

# ler CSV da Raw

df_source = read_source_csv(
    spark        = spark,
    source_path  = SOURCE_PATH,
    adls_options = adls_options,
    csv_options  = {
        "header"      : "true",
        "inferSchema" : "false",
    }
)

df_source.printSchema()
display(df_source.limit(10))

In [0]:
# preparar dados

# contar origem

total_source = df_source.count()
print(f"Total de registros lidos da Raw: {total_source:,}")

In [0]:
# validar colunas esperadas

resultado_colunas = validate_required_columns(
    df               = df_source,
    expected_columns = EXPECTED_COLUMNS
)
print(resultado_colunas["message"])

if resultado_colunas["unexpected_columns"]:
    print(f"Colunas extras: {resultado_colunas['unexpected_columns']}")

In [0]:
# validar chave primaria na origem

resultado_pk = validate_key_columns(
    df          = df_source,
    key_columns = KEY_COLUMNS
)
print(resultado_pk["message"])

In [0]:
# criar DataFrame Bronze
# copia fiel — sem alteracoes nos dados
# adiciona apenas colunas de auditoria

df_bronze = (
    df_source
    .select(
        *EXPECTED_COLUMNS,
        col("_metadata.file_path").alias("bronze_source_file")
    )
    .withColumn("bronze_ingested_at", current_timestamp())
)

# converte todos os campos para string
df_bronze = cast_all_columns_to_string(df_bronze)

print("Schema Bronze:")
df_bronze.printSchema()

display(
    df_bronze
    .select(
        "id_loja",
        "nome_loja",
        "cnpj",
        "estado_loja",
        "bronze_ingested_at",
        "bronze_source_file"
    )
    .limit(10)
)

In [0]:
# validar Bronze antes de gravar

validate_bronze_quality(df_bronze, has_partitions=False)

In [0]:
# gravar Bronze Delta no ADLS
# physical_lojas e dado estatico — sem particao

write_delta(
    df           = df_bronze,
    path         = BRONZE_PATH,
    mode         = BRONZE_WRITE_MODE,
    partition_by = None,
    adls_options = adls_options
)

In [0]:
# ler Bronze gravada para validacao

df_bronze_saved = read_delta(
    spark        = spark,
    path         = BRONZE_PATH,
    adls_options = adls_options
)

df_bronze_saved.printSchema()
display(df_bronze_saved.limit(10))

In [0]:
# validar origem x Bronze

compare_row_counts(
    source_df = df_source,
    target_df = df_bronze_saved,
    label     = "Raw x Bronze physical_lojas"
)

In [0]:
# validar qualidade final da Bronze gravada

validate_bronze_quality(df_bronze_saved, has_partitions=False)

In [0]:
# validar Bronze gravada x Bronze original

print("=" * 55)
print("BRONZE physical_lojas concluida com sucesso!")
print("=" * 55)
print(f"""
   Source Path  : {SOURCE_PATH}
   Bronze Path  : {BRONZE_PATH}
   Total Raw    : {total_source:,}
   Total Bronze : {df_bronze_saved.count():,}
   Particao     : sem particao (dado estatico)
   Status       : SUCESSO
""")